# Broad Obesity Crunch 2 — EDA & Additive Delta Model

Self-contained notebook. Runs on M1 (8GB or 16GB) or Colab.  
Memory-safe: only loads HVG subset (~360MB) into dense arrays.

**Sections**
1. Setup & data paths
2. Data inventory (backed mode — no RAM cost)
3. HVG selection from a 5k-cell sample
4. Per-perturbation means on HVGs
5. Additivity test on known single-gene pairs
6. Constrained least squares for unknown gene deltas
7. Scoring against local ground truth
8. Summary

## 1. Setup

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Data path — edit this one line if needed ─────────────────────────
DATA_DIR = os.path.expanduser(
    '~/ml-explorations/broad_obesity/broad-obesity-2-varying-marsupial/data'
)
# If running on Colab:
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/crunch-data'

TRAIN_H5AD  = f'{DATA_DIR}/obesity_challenge_2.h5ad'
GTRUTH_H5AD = f'{DATA_DIR}/obesity_challenge_2_local_gtruth.h5ad'
PREDICT_TXT = f'{DATA_DIR}/predict_perturbations_2.txt'
PROP_CSV    = f'{DATA_DIR}/program_proportion.csv'

for f in [TRAIN_H5AD, GTRUTH_H5AD, PREDICT_TXT, PROP_CSV]:
    print(f'  {"OK" if os.path.exists(f) else "MISSING":7s}  {os.path.basename(f)}')

## 2. Data inventory (backed mode)

In [ ]:
adata = ad.read_h5ad(TRAIN_H5AD, backed='r')
print(adata)
print(f'\nShape: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(f'Unique perturbations: {adata.obs["gene"].nunique()}')

gene_counts = adata.obs['gene'].value_counts()
single = adata.obs['gene'][~adata.obs['gene'].str.contains(r'\+', regex=True)]
double = adata.obs['gene'][adata.obs['gene'].str.contains(r'\+', regex=True)]
print(f'\nSingle-gene: {single.nunique()} types, {len(single):,} cells')
print(f'Double-gene: {double.nunique()} types, {len(double):,} cells')
print(f'\nCells/perturbation — min:{gene_counts.min()}  '
      f'median:{gene_counts.median():.0f}  max:{gene_counts.max():,}')

with open(PREDICT_TXT) as f:
    predict_perts = [l.strip() for l in f if l.strip()]
print(f'\nPerturbations to predict: {len(predict_perts)}')

# Coverage analysis
single_gene_in_train = [g for g in adata.obs['gene'].unique()
                        if '+' not in g and g != 'NC']
train_perts = set(adata.obs['gene'].unique())
seen   = [p for p in predict_perts if p in train_perts]
unseen = [p for p in predict_perts if p not in train_perts]
print(f'Test perts in training data:  {len(seen)}')
print(f'Test perts completely unseen: {len(unseen)}')

## 3. HVG selection (5k-cell sample)

In [ ]:
np.random.seed(42)
sample_idx = sorted(np.random.choice(adata.n_obs, size=5000, replace=False))
adata_sample = adata[sample_idx].to_memory()
sc.pp.highly_variable_genes(adata_sample, n_top_genes=1000, flavor='seurat')
hvg_names = adata_sample.var_names[adata_sample.var['highly_variable']].tolist()
del adata_sample; gc.collect()
print(f'HVGs selected: {len(hvg_names)}')
print(f'Memory after del sample: peak will be ~{len(hvg_names) * adata.n_obs * 4 / 1e9:.2f} GB for full dense load')

## 4. Per-perturbation means on HVGs

In [ ]:
print('Computing perturbation means on HVGs...')
pert_means_hvg = {}
unique_perts = adata.obs['gene'].unique()

for i, gene in enumerate(unique_perts):
    idx = np.where(adata.obs['gene'].values == gene)[0]
    cells = adata[idx, hvg_names].to_memory()
    X = cells.X
    if sp.issparse(X): X = X.toarray()
    pert_means_hvg[gene] = X.mean(axis=0)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(unique_perts)} done')

print(f'Done. {len(pert_means_hvg)} perturbation means computed.')
nc_mean_hvg = pert_means_hvg['NC']

# Perturbed mean fallback (grand mean of all non-NC perturbations)
perturbed_mean_hvg = np.stack([
    v for k, v in pert_means_hvg.items() if k not in ('NC', 'NC+NC')
]).mean(axis=0)
print('NC mean and perturbed mean fallback ready.')

## 5. Additivity test on known single-gene pairs

In [ ]:
single_genes = ['CEBPB', 'KIF11', 'NR3C1', 'POLR2D',
                'SF3B1', 'STAT5A', 'STAT5B', 'TCF7L2', 'ZBED3']
known_deltas_hvg = {g: pert_means_hvg[g] - nc_mean_hvg for g in single_genes}

print(f'Testing additivity on double-gene pairs where both genes are known:\n')
print(f'{"Perturbation":30s}  {"Pearson r":>10}  {"n_cells":>8}')
print('-' * 55)

results = []
for pert in adata.obs['gene'].unique():
    parts = pert.split('+')
    if len(parts) != 2: continue
    g1, g2 = parts
    if g1 not in known_deltas_hvg or g2 not in known_deltas_hvg: continue
    pred = nc_mean_hvg + known_deltas_hvg[g1] + known_deltas_hvg[g2]
    obs  = pert_means_hvg[pert]
    r, _ = pearsonr(pred, obs)
    n = (adata.obs['gene'] == pert).sum()
    results.append((pert, r, n))
    print(f'{pert:30s}  {r:>10.4f}  {n:>8}')

print(f'\nMean Pearson r: {np.mean([r for _,r,_ in results]):.4f}')
print('>>> Additivity holds well (r > 0.98 expected)')

## 6. Constrained least squares for 9 unknown gene deltas

In [ ]:
# Fix the 9 known deltas, solve only for the 9 unknown ones
unknown_genes = ['CEBPA', 'CEBPD', 'CREB1', 'FOXO1',
                 'KLF15', 'MLXIPL', 'PPARG', 'PPARG2', 'SREBF1']
unknown_to_idx = {g: i for i, g in enumerate(unknown_genes)}

rows, rhs = [], []
for pert, mean_expr in pert_means_hvg.items():
    if pert in ('NC', 'NC+NC'): continue
    parts = [g for g in pert.split('+') if g != 'NC']
    unknowns_in_pert = [g for g in parts if g in unknown_to_idx]
    if not unknowns_in_pert: continue

    residual = mean_expr - nc_mean_hvg
    for g in parts:
        if g in known_deltas_hvg:
            residual -= known_deltas_hvg[g]

    row = np.zeros(len(unknown_genes))
    for g in unknowns_in_pert:
        row[unknown_to_idx[g]] += 1.0

    n_cells = (adata.obs['gene'] == pert).sum()
    rows.append(row * np.sqrt(n_cells))
    rhs.append(residual * np.sqrt(n_cells))

A2 = np.stack(rows)
B2 = np.stack(rhs)
print(f'System: {A2.shape[0]} equations, {A2.shape[1]} unknowns')
print(f'Condition number: {np.linalg.cond(A2):.1f}')

Delta2, _, rank2, _ = np.linalg.lstsq(A2, B2, rcond=None)
print(f'Rank: {rank2} / {len(unknown_genes)}')

# Merge all deltas
all_deltas_hvg = dict(known_deltas_hvg)
for g, idx in unknown_to_idx.items():
    all_deltas_hvg[g] = Delta2[idx]

print(f'\nAll {len(all_deltas_hvg)} gene deltas ready:')
print(sorted(all_deltas_hvg.keys()))

## 7. Score against local ground truth

In [ ]:
# Load ground truth means on HVGs only
print('Loading ground truth means on HVGs...')
gtruth = ad.read_h5ad(GTRUTH_H5AD, backed='r')

gtruth_means_hvg = {}
for gene in gtruth.obs['gene'].unique():
    idx = np.where(gtruth.obs['gene'].values == gene)[0]
    cells = gtruth[idx, hvg_names].to_memory()
    X = cells.X
    if sp.issparse(X): X = X.toarray()
    gtruth_means_hvg[gene] = X.mean(axis=0)

print(f'Done. {len(gtruth_means_hvg)} ground truth perturbations.')

In [ ]:
# Predict and score all 62 test perturbations
def predict(pert, nc_mean, all_deltas, fallback):
    parts = [g for g in pert.split('+') if g != 'NC']
    if not parts: return nc_mean.copy()
    missing = [g for g in parts if g not in all_deltas]
    if missing: return fallback.copy()
    pred = nc_mean.copy()
    for g in parts: pred = pred + all_deltas[g]
    return pred

def tier(pert, known, all_deltas):
    parts = [g for g in pert.split('+') if g != 'NC']
    if all(g in known for g in parts): return 'FULL'
    if any(g in known for g in parts): return 'PARTIAL'
    return 'NONE'

print(f'{"Perturbation":35s}  {"r_additive":>10}  {"r_baseline":>10}  {"tier":>8}')
print('-' * 72)

add_scores, base_scores, tiers = [], [], []
for pert in predict_perts:
    if pert not in gtruth_means_hvg:
        print(f'  MISSING from ground truth: {pert}')
        continue

    obs  = gtruth_means_hvg[pert]
    pred = predict(pert, nc_mean_hvg, all_deltas_hvg, perturbed_mean_hvg)
    base = perturbed_mean_hvg

    r_add,  _ = pearsonr(pred, obs)
    r_base, _ = pearsonr(base, obs)
    t = tier(pert, known_deltas_hvg, all_deltas_hvg)

    add_scores.append(r_add)
    base_scores.append(r_base)
    tiers.append(t)
    print(f'  {pert:35s}  {r_add:>10.4f}  {r_base:>10.4f}  {t:>8}')

print('-' * 72)
print(f'  {"MEAN ALL":35s}  {np.mean(add_scores):>10.4f}  {np.mean(base_scores):>10.4f}')
print(f'\nImprovement over baseline: {np.mean(add_scores) - np.mean(base_scores):+.4f}')

# Per-tier breakdown
print('\nPer-tier breakdown:')
for t in ['FULL', 'PARTIAL', 'NONE']:
    idx = [i for i,x in enumerate(tiers) if x == t]
    if idx:
        add_t  = np.mean([add_scores[i]  for i in idx])
        base_t = np.mean([base_scores[i] for i in idx])
        print(f'  {t:8s} ({len(idx):2d} perts)  additive={add_t:.4f}  baseline={base_t:.4f}  '
              f'delta={add_t-base_t:+.4f}')

## 8. Summary

In [ ]:
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Overall mean Pearson r — additive model : {np.mean(add_scores):.4f}')
print(f'Overall mean Pearson r — baseline       : {np.mean(base_scores):.4f}')
print(f'Net improvement                         : {np.mean(add_scores)-np.mean(base_scores):+.4f}')
print()
print('Interpretation:')
if np.mean(add_scores) - np.mean(base_scores) > 0.05:
    print('  Meaningful improvement — additive delta model is worth submitting.')
    print('  Option B (external gene embeddings) could push further.')
elif np.mean(add_scores) - np.mean(base_scores) > 0.01:
    print('  Modest improvement — marginal over baseline.')
    print('  Option B (external gene embeddings) is the right next step.')
else:
    print('  Negligible improvement — constrained LS deltas are not reliable.')
    print('  Option B (external gene embeddings) is necessary to compete.')
print('=' * 60)